In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime
import time


gold_start_time = time.time()

gold_batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


catalog = "insurance"
schema = "homecredit"


print("Gold Layer Started")
print("Batch ID:", gold_batch_id)
     

Gold Layer Started
Batch ID: 20260627_174641


/home/spark-43d86fe2-c114-4083-b46a-94/.ipykernel/343/command-6880830979953813-4170863995:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  gold_batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [0]:
#Loan KPIs

from pyspark.sql.functions import *

df = spark.table("insurance.homecredit.application_train_silver")

loan_kpi = df.agg(

    count("*").alias("total_applications"),

    sum("AMT_CREDIT").alias("total_credit_amount"),

    avg("AMT_CREDIT").alias("avg_credit"),

    avg("AMT_INCOME_TOTAL").alias("avg_income"),

    avg("AMT_ANNUITY").alias("avg_annuity")

)

loan_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance.homecredit.gold_loan_kpi")

display(loan_kpi)

total_applications,total_credit_amount,avg_credit,avg_income,avg_annuity
307511,1.842070841955E11,599025.9997057016,168797.91929698444,27108.485207033245


In [0]:
#Customer KPIs

customer_kpi = (

    df.groupBy("NAME_INCOME_TYPE")

      .agg(

            count("*").alias("customers"),

            avg("AMT_INCOME_TOTAL").alias("avg_income"),

            avg("AMT_CREDIT").alias("avg_credit")

      )

)

customer_kpi.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_customer_kpi")

display(customer_kpi)

NAME_INCOME_TYPE,customers,avg_income,avg_credit
Working,158774,163169.88922292064,577011.0271140111
State servant,21703,179737.96950559827,669819.3007187947
Commercial associate,71617,202955.32728884206,669913.1228898166
Pensioner,55362,136401.2922732199,542546.0567898558
Unemployed,22,110536.36363636363,764386.3636363636
Student,18,170500.0,510787.5
Businessman,10,652500.0,1228500.0
Maternity leave,5,140400.0,749700.0


In [0]:
#Credit Risk KPIs

risk = (

    df.groupBy("TARGET")

      .agg(

          count("*").alias("customers"),

          avg("AMT_CREDIT").alias("avg_credit"),

          avg("AMT_INCOME_TOTAL").alias("avg_income")

      )

)

risk.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_credit_risk")

display(risk)

TARGET,customers,avg_credit,avg_income
1,24825,557778.527673716,165611.76090634443
0,282686,602648.2820019386,169077.7222658179


In [0]:
#Income Type Analysis

income = (

    df.groupBy("NAME_INCOME_TYPE")

      .agg(

            count("*").alias("applications"),

            avg("AMT_CREDIT").alias("avg_credit"),

            avg("AMT_INCOME_TOTAL").alias("avg_income")

      )

)

income.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_income_analysis")


display(income)

NAME_INCOME_TYPE,applications,avg_credit,avg_income
Working,158774,577011.0271140111,163169.88922292064
State servant,21703,669819.3007187947,179737.96950559827
Commercial associate,71617,669913.1228898166,202955.32728884206
Pensioner,55362,542546.0567898558,136401.2922732199
Unemployed,22,764386.3636363636,110536.36363636363
Student,18,510787.5,170500.0
Businessman,10,1228500.0,652500.0
Maternity leave,5,749700.0,140400.0


In [0]:
#Education Analysis

education = (

    df.groupBy("NAME_EDUCATION_TYPE")

      .agg(

            count("*").alias("applications"),

            avg("AMT_CREDIT").alias("avg_credit"),

            avg("AMT_INCOME_TOTAL").alias("avg_income")

      )

)

education.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_education_analysis")


display(education)

NAME_EDUCATION_TYPE,applications,avg_credit,avg_income
Secondary / secondary special,218391,571193.3924543593,155158.5121375194
Higher education,74863,689950.456099809,208652.05381443442
Incomplete higher,10277,566730.5591612338,181563.8123966138
Lower secondary,3816,489748.5613207547,130079.35849056604
Academic degree,164,723515.625,240009.1463414634


In [0]:
#Occupation Analysis

occupation = (

    df.groupBy("OCCUPATION_TYPE")

      .agg(

            count("*").alias("customers"),

            avg("AMT_INCOME_TOTAL").alias("income"),

            avg("AMT_CREDIT").alias("credit")

      )

)

occupation.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_occupation")

display(occupation)

OCCUPATION_TYPE,customers,income,credit
Laborers,55186,166357.48252518752,570617.9955967093
Core staff,27570,172656.69525408052,625223.1293797606
Accountants,9813,194578.35784163864,709757.3772546622
Managers,21371,260336.68171657855,775091.1945159328
Unknown,96391,153516.03175239387,574329.7325943294
Drivers,18603,187011.60641267538,612333.969037252
Sales staff,32102,152302.8747102984,563258.4920877204
Cleaning staff,4653,130790.89555125726,510960.9497098646
Cooking staff,5946,138396.50817608475,539174.3158425832
Private service staff,2652,182334.81278280544,630664.5033936652


In [0]:
#Regional Analysis

region = (

    df.groupBy("REGION_RATING_CLIENT")

      .agg(

            count("*").alias("customers"),

            avg("AMT_INCOME_TOTAL").alias("income"),

            avg("AMT_CREDIT").alias("credit")

      )

)

region.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_region")

display(region)

REGION_RATING_CLIENT,customers,income,credit
2,226984,161892.5204854748,581610.9937044021
1,32197,242402.25864210952,759990.608861074
3,48330,152194.90108752326,573583.2147113594


In [0]:
#Loan Portfolio

portfolio = (

    df.groupBy("NAME_CONTRACT_TYPE")

      .agg(

            count("*").alias("loans"),

            sum("AMT_CREDIT").alias("credit_amount"),

            avg("AMT_CREDIT").alias("avg_credit")

      )

)

portfolio.write \
.mode("overwrite") \
.saveAsTable("insurance.homecredit.gold_portfolio")


display(portfolio)

NAME_CONTRACT_TYPE,loans,credit_amount,avg_credit
Cash loans,278232,1.747201616955E11,627965.7325379539
Revolving loans,29279,9.4869225E9,324017.9821715223


In [0]:
from pyspark.sql.functions import count, sum, avg, when, col

dashboard = df.agg(

    count("*").alias("applications"),

    sum("AMT_CREDIT").alias("total_credit"),

    avg("AMT_CREDIT").alias("average_credit"),

    avg("AMT_INCOME_TOTAL").alias("average_income"),

    avg("AMT_ANNUITY").alias("average_annuity"),

    sum(
        when(col("TARGET") == 1, 1).otherwise(0)
    ).alias("defaults"),

    sum(
        when(col("TARGET") == 0, 1).otherwise(0)
    ).alias("non_defaults")

)

dashboard.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("insurance.homecredit.gold_dashboard")

display(dashboard)

applications,total_credit,average_credit,average_income,average_annuity,defaults,non_defaults
307511,1.842070841955E11,599025.9997057016,168797.91929698444,27108.485207033245,24825,282686
